# Fine-tune Laya for prompt-injection detection

This notebook keeps raw data and training files in ephemeral `/content`, uploads resumable checkpoints to a private Hugging Face model repo, and never uses Google Drive. Before running it, add `HF_TOKEN` (write access) and `HF_OUTPUT_REPO` (for example `your-org/laya-prompt-injection`) to Colab Secrets and accept the gated source-dataset terms listed in the project README.

In [ ]:
import os, subprocess, torch
from google.colab import userdata

assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_OUTPUT_REPO = userdata.get('HF_OUTPUT_REPO')
assert os.environ['HF_TOKEN'] and HF_OUTPUT_REPO, 'Add HF_TOKEN and HF_OUTPUT_REPO to Colab Secrets'
gpu = torch.cuda.get_device_properties(0)
print(f'{gpu.name}: {gpu.total_memory / 2**30:.1f} GiB')
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
import os, subprocess

PLEASE_REPO = 'https://github.com/shart-cloud/please.git'
PLEASE_REF = '007-Prompt-Injection-Test-Bench'  # Prefer the immutable commit after this change is pushed.
LAYA_COMMIT = '9d955671415fc19f069b9cc998928075c1f255ec'
if not os.path.exists('/content/please'):
    subprocess.run(['git', 'clone', '--filter=blob:none', PLEASE_REPO, '/content/please'], check=True)
subprocess.run(['git', '-C', '/content/please', 'fetch', 'origin', PLEASE_REF], check=True)
subprocess.run(['git', '-C', '/content/please', 'checkout', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run(['pip', 'install', '-q',
    f'git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}',
    'datasets>=4.0,<5', 'huggingface_hub>=0.34,<2', 'safetensors>=0.5,<1',
    'transformers>=4.48,<6', 'scipy>=1.13,<2'], check=True)

## Build the ephemeral, leakage-controlled data pack
The builder queries only a capped set of fresh rows from the large aggregate, then globally deduplicates all sources. This cell can take several minutes.

In [ ]:
import json, os, shutil, subprocess

DATA_DIR = '/content/laya-data'
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
subprocess.run([
    'python', '/content/please/crates/eval/scripts/laya_finetune/prepare_data.py',
    '--repo-root', '/content/please', '--out', DATA_DIR,
], check=True, env=os.environ.copy())
manifest = json.load(open(f'{DATA_DIR}/manifest.json'))
print(json.dumps(manifest['counts'], indent=2))

## Smoke test
Run one small epoch without uploading. This verifies model loading, tokenization, forward/backward passes, calibration, and artifact export before spending a full session.

In [ ]:
import os, shutil, subprocess

SMOKE_DIR = '/content/laya-smoke'
if os.path.exists(SMOKE_DIR):
    shutil.rmtree(SMOKE_DIR)
subprocess.run([
    'python', '/content/please/crates/eval/scripts/laya_finetune/train_laya.py',
    '--data', DATA_DIR, '--output', SMOKE_DIR, '--epochs', '1',
    '--limit-train', '512', '--micro-batch', '4', '--grad-accum', '8', '--no-upload',
], check=True, env=os.environ.copy())

## Full run or resume
Set `RESUME = True` after a disconnected session. Rebuild the deterministic data pack first, then this cell downloads `checkpoint_latest` from the private model repo and continues at the next epoch.

In [ ]:
import os, subprocess

RESUME = False
MICRO_BATCH = 4  # Use 2 on a smaller GPU and raise GRAD_ACCUM to 16.
GRAD_ACCUM = 8
command = [
    'python', '/content/please/crates/eval/scripts/laya_finetune/train_laya.py',
    '--data', DATA_DIR, '--output', '/content/laya-output',
    '--hf-output-repo', HF_OUTPUT_REPO, '--epochs', '4',
    '--micro-batch', str(MICRO_BATCH), '--grad-accum', str(GRAD_ACCUM),
]
if RESUME:
    command.append('--resume')
subprocess.run(command, check=True, env=os.environ.copy())